# Context Compression for RAG

## 1. Project Overview

**Task:** Build a RAG pipeline that retrieves a large set of candidate chunks, **reranks** and **compresses** them, then answers using only the most relevant, condensed evidence.

**Why this matters:** Naive RAG retrieves k chunks and feeds them all to the LLM. This causes three problems:
1. **Diluted context** — irrelevant chunks push the LLM's attention away from the real evidence
2. **Token waste** — you pay for (or fill context windows with) chunks that don't help
3. **"Lost in the middle"** — LLMs underweight information in the middle of long contexts

Context compression solves all three by filtering, reranking, and extracting only the useful parts.

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — LLM for answer generation and LLM-based compression
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — embedding model for retrieval
- `cross-encoder/ms-marco-MiniLM-L-6-v2` — reranker model (via sentence-transformers)
- `ChromaDB` — vector store

> **No API keys required.** Everything runs locally.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Understand **why retrieval alone is not enough** — the over-retrieval problem |
| 2 | Implement **cross-encoder reranking** to reorder chunks by true relevance |
| 3 | Build **extractive compression** — pull only the relevant sentences from each chunk |
| 4 | Build **abstractive compression** — LLM-summarize each chunk to its query-relevant essence |
| 5 | Combine **retrieve → rerank → compress → answer** into a full pipeline |
| 6 | Measure the impact of compression on answer quality and context efficiency |

## 3. What Is Context Compression?

### The Over-Retrieval Problem

```
Naive RAG:        Query → Retrieve top-20 → Feed ALL 20 chunks to LLM → Answer
                                              ↑
                              Maybe 3-5 are actually relevant.
                              The other 15 are noise.

Compressed RAG:   Query → Retrieve top-20 → Rerank → Keep top-5 → Compress → Answer
                                              ↑           ↑             ↑
                                         Better order  Drop noise  Extract key info
```

### Three Levels of Compression

| Level | What It Does | Example |
|-------|-------------|--------|
| **Filter** | Drop entire chunks scored below a threshold | 20 chunks → 5 chunks |
| **Rerank** | Reorder by true query relevance (not just embedding similarity) | Chunk #17 moves to #1 |
| **Extract** | Keep only relevant sentences from each chunk | 200-word chunk → 40-word extract |
| **Abstract** | LLM-summarize each chunk relative to the query | 200-word chunk → 30-word summary |

### Why Embedding Retrieval Ranking Is Not Enough

Bi-encoder embeddings (like MiniLM) encode query and document **independently**. They miss:
- Word-level interactions between query and document
- Negation ("does NOT support X" matches "supports X" similarly)
- Specificity (a generic chunk about databases ranks near a specific PostgreSQL question)

Cross-encoders process query + document **together**, capturing fine-grained interactions.

## 4. Pipeline Architecture

```
  Question
     │
     ▼
  ┌───────────────┐
  │ 1. RETRIEVE   │  Bi-encoder (fast, approximate)
  │    top-20     │  Casts wide net: high recall, lower precision
  └───────┬───────┘
          │ 20 chunks
          ▼
  ┌───────────────┐
  │ 2. RERANK     │  Cross-encoder (slower, more accurate)
  │    top-5      │  Reorders by true relevance
  └───────┬───────┘
          │ 5 chunks
          ▼
  ┌───────────────┐
  │ 3. COMPRESS   │  Extractive or Abstractive
  │    condense   │  Keep only query-relevant information
  └───────┬───────┘
          │ ~30% of original text
          ▼
  ┌───────────────┐
  │ 4. GENERATE   │  LLM answers with clean, focused context
  │    answer     │  Less noise → better answers
  └───────────────┘
```

## 5. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers

print("Dependencies: langchain, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")
print("Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2 (auto-downloaded)")

## 6. Imports

## 7. Configuration

In [ ]:
import os
import re
import json
import time
import textwrap
import warnings
from collections import Counter

import numpy as np

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder
import chromadb

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
RETRIEVE_K = 15       # Cast a wide net
RERANK_TOP = 5        # Keep only the best after reranking
TEMP_ANSWER = 0.2
TEMP_COMPRESS = 0.0   # Deterministic for compression

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Reranker: {RERANK_MODEL}")
print(f"Retrieve top-{RETRIEVE_K} → Rerank to top-{RERANK_TOP}")

## 8. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


def word_count(text: str) -> int:
    return len(text.split())


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Knowledge Base & Retriever

## 9. Document Corpus

30 passages on software engineering topics — deliberately varied in length and relevance density so compression has something to work with.

In [ ]:
PASSAGES = [
    {"id": "P01", "topic": "docker",
     "text": "Docker containers package applications with their dependencies into standardized units that share the host OS kernel. Unlike virtual machines, containers do not need a full guest operating system — they reuse the host kernel via Linux namespaces and cgroups. Docker images are built from Dockerfiles using a layered union file system. Each instruction (FROM, RUN, COPY) creates a new read-only layer. At runtime, a thin writable layer is added on top. This layered approach means shared base images save significant disk space across many containers. Containers typically start in under 2 seconds and consume 50-100 MB of memory."},
    {"id": "P02", "topic": "docker",
     "text": "Docker Compose defines multi-container applications using a YAML file. Services, networks, and volumes are declared together. The 'docker compose up' command starts all services. Docker Compose is ideal for development environments where multiple services need to communicate. It supports environment variables, dependency ordering with 'depends_on', and health checks. Docker Compose V2 is written in Go and integrated directly into the Docker CLI as a plugin."},
    {"id": "P03", "topic": "docker",
     "text": "Docker networking allows containers to communicate. Bridge networks provide isolation on a single host. Overlay networks span multiple Docker hosts for swarm services. Host networking removes network isolation, giving the container direct access to host ports. Published ports map container ports to host ports using '-p host:container'. DNS-based service discovery lets containers resolve each other by name within user-defined networks."},
    {"id": "P04", "topic": "kubernetes",
     "text": "Kubernetes (K8s) orchestrates containerized applications across clusters of machines. The control plane includes the API server, scheduler, controller manager, and etcd. Worker nodes run kubelet, kube-proxy, and a container runtime. A Pod is the smallest deployable unit — one or more containers sharing network and storage. Deployments manage desired state: specifying 3 replicas means K8s ensures exactly 3 pods run at all times. If a pod crashes, the deployment controller automatically creates a replacement."},
    {"id": "P05", "topic": "kubernetes",
     "text": "Kubernetes Services provide stable network endpoints for sets of pods. ClusterIP (default) exposes the service on an internal IP. NodePort exposes on each node's IP at a static port. LoadBalancer provisions an external load balancer from the cloud provider. Ingress resources manage external HTTP/HTTPS routing with path-based and host-based rules. Service meshes like Istio add mTLS, traffic shaping, and observability at the network layer."},
    {"id": "P06", "topic": "kubernetes",
     "text": "Kubernetes ConfigMaps store non-sensitive configuration as key-value pairs, mountable as environment variables or files. Secrets store sensitive data like passwords and tokens, base64-encoded at rest. Horizontal Pod Autoscaler adjusts replica count based on CPU, memory, or custom metrics. Vertical Pod Autoscaler adjusts resource requests and limits. The Cluster Autoscaler adds or removes nodes based on pending pod demands."},
    {"id": "P07", "topic": "databases",
     "text": "PostgreSQL uses MVCC (Multi-Version Concurrency Control) so readers never block writers and vice versa. Each transaction sees a consistent snapshot of the database. Write-Ahead Logging (WAL) ensures durability — changes are written to the WAL before being applied to data files. If the server crashes, PostgreSQL replays the WAL to recover. The default index type is B-tree, optimal for equality and range queries. GiST indexes support full-text search, PostGIS geometry, and nearest-neighbor queries."},
    {"id": "P08", "topic": "databases",
     "text": "Database indexing strategies vary by workload. B-tree indexes handle equality and range lookups efficiently but slow down writes because every insert must update the index. Hash indexes are faster for exact equality but cannot support range queries. Partial indexes cover only rows matching a condition, saving storage for sparse queries. Covering indexes include all columns needed by a query, enabling index-only scans that avoid accessing the main table."},
    {"id": "P09", "topic": "databases",
     "text": "Database sharding partitions data across multiple servers. Hash-based sharding distributes rows by hashing the shard key — this gives even distribution but makes range queries expensive since they must hit all shards. Range-based sharding partitions by value ranges — better for range queries but can create hot spots if some ranges are more popular. Adding new shards requires careful data rebalancing. Cross-shard joins require scatter-gather operations which add latency."},
    {"id": "P10", "topic": "caching",
     "text": "Redis is an in-memory data structure store achieving sub-millisecond latency. It supports strings, lists, sets, sorted sets, hashes, streams, and HyperLogLog. Redis persistence options: RDB creates point-in-time snapshots at configurable intervals; AOF logs every write operation for durability. Redis Sentinel monitors instances and performs automatic failover. Redis Cluster distributes data across multiple nodes with hash slots, supporting up to 16384 hash slots partitioned across the cluster."},
    {"id": "P11", "topic": "caching",
     "text": "Cache invalidation is widely considered one of the two hardest problems in computer science. Time-to-live (TTL) automatically expires entries after a fixed period — simple but can serve stale data during the TTL window. Write-through updates the cache and database simultaneously, ensuring consistency but adding write latency. Write-behind batches database writes asynchronously for better write performance but risks data loss on cache failure. Cache-aside (lazy loading) populates cache only on a miss."},
    {"id": "P12", "topic": "caching",
     "text": "The cache stampede (thundering herd) problem occurs when a popular key expires and many concurrent requests simultaneously try to regenerate it, overwhelming the backend. Solutions include: mutex locking (only one request regenerates while others wait), probabilistic early expiration (XFetch algorithm refreshes before TTL), and background refresh (a separate process preemptively updates popular keys). Cache warming pre-populates the cache during deployment to avoid cold-start misses."},
    {"id": "P13", "topic": "api",
     "text": "REST APIs use HTTP methods semantically: GET reads (safe, idempotent), POST creates (not idempotent), PUT replaces entire resources (idempotent), PATCH partially updates, DELETE removes. Resources are identified by URIs as nouns, not verbs. Status codes: 2xx success (200 OK, 201 Created, 204 No Content), 4xx client errors (400 Bad Request, 401 Unauthorized, 404 Not Found, 409 Conflict), 5xx server errors (500, 503 Service Unavailable)."},
    {"id": "P14", "topic": "api",
     "text": "API rate limiting protects services from abuse. Token bucket allows bursts up to a fixed bucket size then limits to a steady rate. Sliding window counts requests in a moving time window. Rate limits are communicated via headers: X-RateLimit-Limit (max), X-RateLimit-Remaining (left), X-RateLimit-Reset (when limit resets). Exceeding the limit returns 429 Too Many Requests. API keys identify callers; different tiers can have different rate limits."},
    {"id": "P15", "topic": "api",
     "text": "GraphQL provides a single endpoint where clients specify exactly what data they need through typed queries. The schema defines types, fields, and relationships using SDL. Resolvers are functions that fetch data for each field. Mutations handle writes. Subscriptions enable real-time updates via WebSockets. The main advantage over REST is eliminating over-fetching and under-fetching. However, caching is harder because every query string is unique."},
    {"id": "P16", "topic": "security",
     "text": "OAuth 2.0 delegates authorization without sharing credentials. The authorization code flow: the client redirects the user to the authorization server, the user grants permission, receives a code, which the client exchanges for tokens. Access tokens are short-lived (typically 1 hour). Refresh tokens are long-lived and used to obtain new access tokens. PKCE (Proof Key for Code Exchange) prevents authorization code interception in public clients like mobile apps."},
    {"id": "P17", "topic": "security",
     "text": "JWT (JSON Web Tokens) encode claims as a JSON payload signed with HMAC or RSA. The three parts: header (algorithm), payload (claims like sub, exp, iat), signature. JWTs are stateless — the server doesn't need to store session data. Drawbacks: tokens cannot be revoked before expiration without a blacklist, token size can be large, and sensitive data should never be stored in the payload since it's only base64-encoded (not encrypted)."},
    {"id": "P18", "topic": "security",
     "text": "SQL injection occurs when user input is concatenated into queries. Prevention: parameterized queries (prepared statements) separate SQL logic from data. ORMs generally prevent injection by using parameterized queries internally. Cross-site scripting (XSS) injects malicious scripts into web pages. Stored XSS persists in the database. Reflected XSS bounces input from the URL. Content Security Policy (CSP) restricts script sources. Input validation and output encoding are defense-in-depth measures."},
    {"id": "P19", "topic": "messaging",
     "text": "Apache Kafka is a distributed event streaming platform handling millions of events per second. Producers publish records to topics. Topics are partitioned across brokers for parallel processing. Each partition is an ordered, append-only log. Consumer groups enable parallel consumption — each partition is assigned to exactly one consumer in the group. Kafka retains messages for a configurable period (default 7 days) regardless of whether they've been consumed, enabling replay."},
    {"id": "P20", "topic": "messaging",
     "text": "Message delivery guarantees: at-most-once (fire and forget — messages may be lost), at-least-once (messages are retried — duplicates possible), exactly-once (hardest to achieve — requires idempotent processing or transactional coordination). Kafka supports exactly-once semantics within its ecosystem using idempotent producers and transactional APIs. Dead letter queues capture messages that fail processing after maximum retry attempts for later investigation."},
    {"id": "P21", "topic": "monitoring",
     "text": "The four golden signals of monitoring (from Google SRE): latency (response time), traffic (requests per second), errors (error rate), and saturation (resource utilization). Prometheus collects time-series metrics using a pull model — it scrapes HTTP /metrics endpoints on a schedule. Counter metrics only increase (request count). Gauge metrics go up and down (temperature, queue size). Histogram metrics track value distributions (response time buckets)."},
    {"id": "P22", "topic": "monitoring",
     "text": "Distributed tracing follows requests across microservice boundaries. Each request gets a unique trace ID. Each service call within that request is a span with start time, duration, and metadata. Parent-child span relationships form a call tree. OpenTelemetry is the CNCF standard providing APIs and SDKs for traces, metrics, and logs. Jaeger and Zipkin are popular tracing backends. Traces help identify latency bottlenecks across service boundaries."},
    {"id": "P23", "topic": "architecture",
     "text": "Microservices decompose applications into independently deployable services, each owning its data store. Benefits: independent deployment and scaling, fault isolation, technology diversity per service. Challenges: distributed transactions (use sagas for eventual consistency), service discovery, operational complexity. Communication patterns: synchronous REST/gRPC calls or asynchronous messaging via events. The strangler fig pattern incrementally migrates monolith functionality to microservices."},
    {"id": "P24", "topic": "architecture",
     "text": "CQRS (Command Query Responsibility Segregation) separates read and write models. Commands modify state and produce events. Queries read from optimized projections (denormalized read models). Event sourcing stores every state change as an immutable event rather than overwriting current state. The current state is derived by replaying all events. Benefits: complete audit trail, temporal queries, event-driven integrations. Challenges: eventual consistency between write and read models, event schema evolution."},
    {"id": "P25", "topic": "architecture",
     "text": "The CAP theorem states distributed systems cannot simultaneously guarantee Consistency, Availability, and Partition tolerance — during a network partition, you must choose CP or AP. In practice, PACELC extends this: even without partitions, there's a trade-off between Latency and Consistency. CP systems (HBase, MongoDB with majority reads) reject requests to maintain consistency. AP systems (Cassandra, DynamoDB) accept all writes, risking conflicts resolved via last-write-wins or vector clocks."},
    {"id": "P26", "topic": "ci_cd",
     "text": "CI/CD pipelines automate the path from code commit to production. Continuous Integration runs automated tests on every commit: linting, unit tests, integration tests, security scanning (SAST/DAST). Continuous Delivery ensures the main branch is always deployable. Continuous Deployment automatically pushes every passed commit to production. GitHub Actions defines workflows in YAML triggered by events (push, pull_request, schedule). Matrix builds test across multiple OS/language versions."},
    {"id": "P27", "topic": "ci_cd",
     "text": "Deployment strategies control how new versions roll out. Rolling updates replace pods gradually — some traffic goes to old version, some to new. Blue-green deployment maintains two identical environments; traffic switches from blue to green after verification. Canary deployment routes 5-10% of traffic initially; if metrics are healthy, percentage increases. Feature flags decouple deployment from release — code is deployed but toggled off until ready."},
    {"id": "P28", "topic": "testing",
     "text": "The testing pyramid recommends: many fast unit tests (test individual functions in isolation using mocks), fewer integration tests (verify component interactions with real databases and APIs), and minimal end-to-end tests (simulate full user workflows through the UI). Contract testing verifies API compatibility between services without running full integration tests. Property-based testing generates random inputs and checks invariants — useful for finding edge cases that example tests miss."},
    {"id": "P29", "topic": "testing",
     "text": "Test-Driven Development (TDD) follows red-green-refactor: write a failing test first, implement the minimum code to pass, then refactor. Behavior-Driven Development (BDD) uses Given-When-Then syntax readable by non-technical stakeholders. Mutation testing modifies source code (introduces bugs) to verify that tests detect the change — if a mutation survives (tests still pass), test coverage is inadequate for that code path."},
    {"id": "P30", "topic": "networking",
     "text": "Load balancers distribute traffic across backend servers for scalability and fault tolerance. Layer 4 (transport) balancers route by IP/TCP without inspecting payload — fast but limited. Layer 7 (application) balancers inspect HTTP headers and URLs for content-based routing, SSL termination, and request modification. Algorithms: round-robin (simple rotation), least connections (route to least busy), weighted (prefer more capable servers), IP hash (session affinity)."},
]

print(f"Corpus: {len(PASSAGES)} passages, {len(set(p['topic'] for p in PASSAGES))} topics")
total_words = sum(word_count(p['text']) for p in PASSAGES)
print(f"Total words: {total_words}, avg per passage: {total_words // len(PASSAGES)}")

## 10. Build the Vector Store & Retriever

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("kb")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="kb", metadata={"hnsw:space": "cosine"},
)

texts = [p["text"] for p in PASSAGES]
metas = [{"passage_id": p["id"], "topic": p["topic"]} for p in PASSAGES]
ids = [p["id"] for p in PASSAGES]
vecs = embeddings.embed_documents(texts)
collection.add(documents=texts, embeddings=vecs, metadatas=metas, ids=ids)

print(f"Indexed {collection.count()} passages")


def retrieve(query: str, top_k: int = RETRIEVE_K) -> list[dict]:
    """Retrieve top-k chunks by embedding similarity."""
    qv = embeddings.embed_query(query)
    results = collection.query(query_embeddings=[qv], n_results=top_k)
    hits = []
    for i in range(len(results["documents"][0])):
        hits.append({
            "passage_id": results["metadatas"][0][i]["passage_id"],
            "topic": results["metadatas"][0][i]["topic"],
            "text": results["documents"][0][i],
            "embed_score": round(1 - results["distances"][0][i], 4),
        })
    return hits


# Quick test
test_hits = retrieve("How does Redis work?", top_k=5)
print(f"\nTest retrieval for 'How does Redis work?':")
for h in test_hits:
    print(f"  [{h['passage_id']}] ({h['topic']}) embed_score={h['embed_score']:.3f} | {h['text'][:50]}...")

---

# Part B — Reranking

## 11. Load the Cross-Encoder Reranker

A **cross-encoder** processes the (query, document) pair together through a single transformer, producing a relevance score. This is slower than bi-encoder retrieval but much more accurate because it can attend to word-level interactions.

```
Bi-encoder (retrieval):    encode(query) · encode(doc) → similarity
                           Fast, but query and doc are encoded independently

Cross-encoder (reranking): encode(query + " [SEP] " + doc) → relevance score
                           Slower, but sees both together
```

In [ ]:
reranker = CrossEncoder(RERANK_MODEL)


def rerank(query: str, passages: list[dict], top_n: int = RERANK_TOP) -> list[dict]:
    """Rerank passages using cross-encoder, return top-n."""
    pairs = [(query, p["text"]) for p in passages]
    scores = reranker.predict(pairs)

    for p, score in zip(passages, scores):
        p["rerank_score"] = float(score)

    ranked = sorted(passages, key=lambda x: x["rerank_score"], reverse=True)
    return ranked[:top_n]


print(f"Cross-encoder reranker loaded: {RERANK_MODEL}")
print(f"This model processes (query, doc) pairs jointly for accurate relevance scoring.")

## 12. Reranking Demo — See How Rankings Change

In [ ]:
demo_query = "How does cache invalidation work and what problems can occur?"

# Retrieve top-15 by embedding
demo_retrieved = retrieve(demo_query, top_k=15)

# Rerank to top-5
demo_reranked = rerank(demo_query, demo_retrieved, top_n=5)

print(f"Query: {demo_query}")
print(f"\nBEFORE RERANKING (top-8 by embedding similarity):")
print(f"{'Rank':>4} {'ID':>5} {'Topic':<12} {'Embed':>7} | Preview")
print("-" * 85)
for i, p in enumerate(demo_retrieved[:8], 1):
    print(f"  {i:>2}  [{p['passage_id']}] {p['topic']:<12} {p['embed_score']:>6.3f} | {p['text'][:45]}...")

print(f"\nAFTER RERANKING (top-5 by cross-encoder):")
print(f"{'Rank':>4} {'ID':>5} {'Topic':<12} {'Rerank':>7} {'Embed':>7} | Preview")
print("-" * 95)
for i, p in enumerate(demo_reranked, 1):
    print(f"  {i:>2}  [{p['passage_id']}] {p['topic']:<12} {p['rerank_score']:>7.3f} {p['embed_score']:>6.3f} | {p['text'][:40]}...")

# Show which passages changed rank
embed_order = [p["passage_id"] for p in demo_retrieved[:5]]
rerank_order = [p["passage_id"] for p in demo_reranked]
promoted = set(rerank_order) - set(embed_order)
demoted = set(embed_order) - set(rerank_order)
if promoted:
    print(f"\n  PROMOTED into top-5: {promoted}")
if demoted:
    print(f"  DEMOTED from top-5: {demoted}")

---

# Part C — Context Compression

## 13. Strategy 1 — Extractive Compression

**Idea:** Each passage contains many sentences, but only some are relevant to the query. Extractive compression keeps only the relevant sentences and discards the rest.

```
PASSAGE (85 words):
  "Redis is an in-memory data structure store achieving sub-millisecond
   latency. It supports strings, lists, sets [... more types ...]
   Redis persistence options: RDB creates point-in-time snapshots [...].
   Redis Sentinel monitors instances and performs automatic failover.
   Redis Cluster distributes data across nodes [...]"

QUERY: "What persistence options does Redis have?"

EXTRACTED (20 words):
  "Redis persistence options: RDB creates point-in-time snapshots at
   configurable intervals; AOF logs every write operation for durability."
```

In [ ]:
def extractive_compress(query: str, passages: list[dict], sim_threshold: float = 0.3) -> list[dict]:
    """Keep only sentences with high similarity to the query."""
    query_vec = np.array(embeddings.embed_query(query))
    compressed = []

    for p in passages:
        # Split into sentences
        sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', p["text"]) if len(s.strip()) > 10]
        if not sents:
            continue

        # Score each sentence
        sent_vecs = embeddings.embed_documents(sents)
        kept = []
        for sent, svec in zip(sents, sent_vecs):
            sv = np.array(svec)
            sim = float(np.dot(query_vec, sv) / (np.linalg.norm(query_vec) * np.linalg.norm(sv)))
            if sim >= sim_threshold:
                kept.append({"sentence": sent, "similarity": sim})

        if kept:
            kept.sort(key=lambda x: x["similarity"], reverse=True)
            compressed.append({
                "passage_id": p["passage_id"],
                "topic": p["topic"],
                "original_words": word_count(p["text"]),
                "compressed_text": " ".join(k["sentence"] for k in kept),
                "compressed_words": sum(word_count(k["sentence"]) for k in kept),
                "sentences_kept": len(kept),
                "sentences_total": len(sents),
            })

    return compressed


# Demo
demo_q = "What persistence options does Redis have?"
demo_p = retrieve(demo_q, top_k=5)
demo_reranked = rerank(demo_q, demo_p, top_n=3)
demo_compressed = extractive_compress(demo_q, demo_reranked)

print(f"Query: {demo_q}")
print(f"\nEXTRACTIVE COMPRESSION RESULTS")
print("=" * 90)

for c in demo_compressed:
    ratio = c["compressed_words"] / c["original_words"] * 100
    print(f"  [{c['passage_id']}] {c['original_words']} words → {c['compressed_words']} words ({ratio:.0f}%)")
    print(f"        Sentences: {c['sentences_kept']}/{c['sentences_total']} kept")
    print(f"        Text: {c['compressed_text'][:120]}...")
    print()

## 14. Strategy 2 — Abstractive Compression (LLM-Based)

**Idea:** Instead of selecting sentences, ask the LLM to summarize each passage **relative to the query**, extracting only the query-relevant information.

In [ ]:
COMPRESS_SYSTEM = """You compress text by extracting ONLY the information relevant to
the query. Remove everything else. Be concise but preserve key facts and numbers."""

COMPRESS_PROMPT = """Given the query, compress this passage to ONLY the relevant information.
If nothing is relevant, respond with just: "NOT RELEVANT"

QUERY: {query}

PASSAGE [{pid}]:
{text}

Compressed (keep only query-relevant facts):"""


def abstractive_compress(query: str, passages: list[dict]) -> list[dict]:
    """LLM-based abstractive compression: summarize each passage for the query."""
    compressed = []
    for p in passages:
        resp = ask(
            COMPRESS_PROMPT.format(query=query, pid=p["passage_id"], text=p["text"]),
            system=COMPRESS_SYSTEM,
            temperature=TEMP_COMPRESS,
        )

        if "NOT RELEVANT" in resp.upper():
            continue

        compressed.append({
            "passage_id": p["passage_id"],
            "topic": p["topic"],
            "original_words": word_count(p["text"]),
            "compressed_text": resp,
            "compressed_words": word_count(resp),
        })

    return compressed


# Demo
demo_abstract = abstractive_compress(demo_q, demo_reranked)

print(f"Query: {demo_q}")
print(f"\nABSTRACTIVE COMPRESSION RESULTS")
print("=" * 90)
for c in demo_abstract:
    ratio = c["compressed_words"] / c["original_words"] * 100
    print(f"  [{c['passage_id']}] {c['original_words']} words → {c['compressed_words']} words ({ratio:.0f}%)")
    print(f"        {c['compressed_text'][:120]}")
    print()

## 15. Strategy 3 — Relevance Filtering Only

The simplest "compression": just drop passages below a reranking score threshold.

In [ ]:
def filter_by_score(passages: list[dict], min_score: float = 0.0) -> list[dict]:
    """Drop passages below the reranking score threshold."""
    kept = [p for p in passages if p.get("rerank_score", 0) >= min_score]
    dropped = len(passages) - len(kept)
    return kept, dropped


# Demo: retrieve 15, see score distribution
demo_all = retrieve(demo_q, top_k=15)
demo_all_ranked = rerank(demo_q, demo_all, top_n=15)

print(f"Query: {demo_q}")
print(f"\nReraning scores for all 15 retrieved passages:")
for i, p in enumerate(demo_all_ranked, 1):
    bar = "#" * int(max(0, p['rerank_score'] + 5) * 3)
    print(f"  {i:>2}. [{p['passage_id']}] score={p['rerank_score']:>7.3f} {bar}")

# Apply threshold
kept, dropped = filter_by_score(demo_all_ranked, min_score=0.0)
print(f"\n  Threshold 0.0: kept {len(kept)}, dropped {dropped}")

kept2, dropped2 = filter_by_score(demo_all_ranked, min_score=2.0)
print(f"  Threshold 2.0: kept {len(kept2)}, dropped {dropped2}")

---

# Part D — Full Pipeline & Comparison

## 16. Build All Pipeline Variants

We'll compare four approaches:
1. **Naive RAG** — retrieve top-5, feed directly to LLM
2. **Reranked RAG** — retrieve top-15, rerank to top-5, feed to LLM
3. **Extractive Compressed RAG** — retrieve top-15, rerank to top-5, extract relevant sentences
4. **Abstractive Compressed RAG** — retrieve top-15, rerank to top-5, LLM-compress each passage

In [ ]:
ANSWER_SYSTEM = """You answer technical questions using ONLY the provided context.
Cite sources as [PXX]. If the context lacks information, say so."""


def generate_answer(question: str, context_text: str) -> str:
    prompt = f"QUESTION: {question}\n\nCONTEXT:\n{context_text}\n\nAnswer:"
    return ask(prompt, system=ANSWER_SYSTEM, temperature=TEMP_ANSWER)


def pipeline_naive(question: str) -> dict:
    """Retrieve top-5, answer directly."""
    t0 = time.time()
    passages = retrieve(question, top_k=5)
    context = "\n\n".join(f"[{p['passage_id']}] {p['text']}" for p in passages)
    answer = generate_answer(question, context)
    return {
        "method": "naive",
        "answer": answer,
        "passages_used": [p["passage_id"] for p in passages],
        "context_words": word_count(context),
        "time_s": time.time() - t0,
    }


def pipeline_reranked(question: str) -> dict:
    """Retrieve top-15, rerank to top-5, answer."""
    t0 = time.time()
    passages = retrieve(question, top_k=15)
    reranked = rerank(question, passages, top_n=5)
    context = "\n\n".join(f"[{p['passage_id']}] {p['text']}" for p in reranked)
    answer = generate_answer(question, context)
    return {
        "method": "reranked",
        "answer": answer,
        "passages_used": [p["passage_id"] for p in reranked],
        "context_words": word_count(context),
        "time_s": time.time() - t0,
    }


def pipeline_extractive(question: str) -> dict:
    """Retrieve top-15, rerank to top-5, extractive compression, answer."""
    t0 = time.time()
    passages = retrieve(question, top_k=15)
    reranked = rerank(question, passages, top_n=5)
    compressed = extractive_compress(question, reranked)
    context = "\n\n".join(f"[{c['passage_id']}] {c['compressed_text']}" for c in compressed)
    answer = generate_answer(question, context)
    return {
        "method": "extractive",
        "answer": answer,
        "passages_used": [c["passage_id"] for c in compressed],
        "context_words": word_count(context),
        "compression_details": compressed,
        "time_s": time.time() - t0,
    }


def pipeline_abstractive(question: str) -> dict:
    """Retrieve top-15, rerank to top-5, abstractive compression, answer."""
    t0 = time.time()
    passages = retrieve(question, top_k=15)
    reranked = rerank(question, passages, top_n=5)
    compressed = abstractive_compress(question, reranked)
    context = "\n\n".join(f"[{c['passage_id']}] {c['compressed_text']}" for c in compressed)
    answer = generate_answer(question, context)
    return {
        "method": "abstractive",
        "answer": answer,
        "passages_used": [c["passage_id"] for c in compressed],
        "context_words": word_count(context),
        "compression_details": compressed,
        "time_s": time.time() - t0,
    }


print("Four pipeline variants ready:")
print("  1. naive        — retrieve 5, answer")
print("  2. reranked     — retrieve 15 → rerank to 5 → answer")
print("  3. extractive   — retrieve 15 → rerank 5 → extract sentences → answer")
print("  4. abstractive  — retrieve 15 → rerank 5 → LLM compress → answer")

## 17. Run Side-by-Side Comparison

In [ ]:
comparison_q = "How does Kubernetes handle automatic scaling and what are the different autoscaler types?"

print(f"Q: {comparison_q}")
print("=" * 100)

results = {}
for name, fn in [
    ("naive", pipeline_naive),
    ("reranked", pipeline_reranked),
    ("extractive", pipeline_extractive),
    ("abstractive", pipeline_abstractive),
]:
    r = fn(comparison_q)
    results[name] = r
    print(f"\n{'─'*100}")
    print(f"METHOD: {name.upper()}")
    print(f"  Passages: {r['passages_used']}")
    print(f"  Context:  {r['context_words']} words")
    print(f"  Time:     {r['time_s']:.1f}s")
    print(f"  Answer:")
    wrap_print("    " + r["answer"][:250] + ("..." if len(r["answer"]) > 250 else ""))

In [ ]:
# Context efficiency comparison
print("CONTEXT EFFICIENCY COMPARISON")
print("=" * 70)
print(f"{'Method':<15} {'Context Words':>14} {'vs Naive':>10} {'Time':>8}")
print("-" * 70)

naive_words = results["naive"]["context_words"]
for name in ["naive", "reranked", "extractive", "abstractive"]:
    r = results[name]
    ratio = r["context_words"] / naive_words * 100
    print(f"  {name:<13} {r['context_words']:>12} {ratio:>9.0f}% {r['time_s']:>7.1f}s")

if "extractive" in results and "compression_details" in results["extractive"]:
    print(f"\n  EXTRACTIVE COMPRESSION DETAIL")
    for c in results["extractive"]["compression_details"]:
        ratio = c["compressed_words"] / c["original_words"] * 100
        print(f"    [{c['passage_id']}] {c['original_words']}→{c['compressed_words']} words "
              f"({ratio:.0f}%) — {c.get('sentences_kept','?')}/{c.get('sentences_total','?')} sentences")

## 18. LLM-as-Judge — Quality Comparison

In [ ]:
JUDGE_SYSTEM = """You compare two answers to the same question.
Judge which answer is more accurate, complete, and well-supported."""

JUDGE_PROMPT = """QUESTION: {question}

ANSWER A ({method_a}):
{answer_a}

ANSWER B ({method_b}):
{answer_b}

Which answer is better? Return ONLY JSON:
{{"winner": "A" or "B" or "tie",
  "reasoning": "brief explanation",
  "quality_a": 1-5,
  "quality_b": 1-5}}"""


def judge_answers(question: str, result_a: dict, result_b: dict) -> dict:
    resp = ask(
        JUDGE_PROMPT.format(
            question=question,
            method_a=result_a["method"],
            answer_a=result_a["answer"][:400],
            method_b=result_b["method"],
            answer_b=result_b["answer"][:400],
        ),
        system=JUDGE_SYSTEM,
        temperature=0.0,
    )
    result = parse_json(resp)
    return result or {"winner": "tie", "reasoning": "Could not parse"}


# Compare: naive vs reranked
print("HEAD-TO-HEAD COMPARISONS")
print("=" * 80)

pairs = [
    ("naive", "reranked"),
    ("naive", "extractive"),
    ("naive", "abstractive"),
    ("reranked", "abstractive"),
]

for ma, mb in pairs:
    j = judge_answers(comparison_q, results[ma], results[mb])
    winner_name = ma if j.get("winner") == "A" else mb if j.get("winner") == "B" else "tie"
    print(f"  {ma:>12} vs {mb:<12} → winner: {winner_name}")
    print(f"    quality: {ma}={j.get('quality_a','?')}/5, {mb}={j.get('quality_b','?')}/5")
    print(f"    reason:  {j.get('reasoning','?')[:70]}")
    print()

---

# Part E — Multi-Question Evaluation

## 19. Evaluation Question Set

In [ ]:
EVAL_QUESTIONS = [
    "What cache invalidation strategies exist and what are their tradeoffs?",
    "How does PostgreSQL ensure data durability after a crash?",
    "What deployment strategies minimize risk when releasing new software versions?",
    "How do microservices handle distributed transactions?",
    "What is the difference between Layer 4 and Layer 7 load balancing?",
    "How does OAuth 2.0 authorization code flow work?",
    "What types of database indexes exist and when to use each?",
    "How does distributed tracing help debug microservice latency?",
]

print(f"Evaluation: {len(EVAL_QUESTIONS)} questions")

## 20. Run Full Evaluation

In [ ]:
print("Running evaluation across all pipelines (LLM-intensive)...\n")

eval_results = []

for i, q in enumerate(EVAL_QUESTIONS, 1):
    row = {"question": q, "results": {}}

    # Run all 4 pipelines
    for name, fn in [
        ("naive", pipeline_naive),
        ("reranked", pipeline_reranked),
        ("extractive", pipeline_extractive),
        ("abstractive", pipeline_abstractive),
    ]:
        r = fn(q)
        row["results"][name] = r

    eval_results.append(row)

    # Brief progress
    ctx = {name: r["context_words"] for name, r in row["results"].items()}
    print(f"  [{i}/{len(EVAL_QUESTIONS)}] {q[:50]}...")
    print(f"           Context words: naive={ctx['naive']}, reranked={ctx['reranked']}, "
          f"extractive={ctx['extractive']}, abstractive={ctx['abstractive']}")

## 21. Aggregate Statistics

In [ ]:
print("AGGREGATE RESULTS")
print("=" * 80)

methods = ["naive", "reranked", "extractive", "abstractive"]

print(f"\n{'Method':<15} {'Avg Words':>10} {'Compression':>12} {'Avg Time':>10}")
print("-" * 55)

stats = {}
for m in methods:
    ctx_words = [row["results"][m]["context_words"] for row in eval_results]
    times = [row["results"][m]["time_s"] for row in eval_results]
    avg_words = sum(ctx_words) / len(ctx_words)
    avg_time = sum(times) / len(times)
    stats[m] = {"avg_words": avg_words, "avg_time": avg_time}

naive_avg = stats["naive"]["avg_words"]

for m in methods:
    compression = stats[m]["avg_words"] / naive_avg * 100
    print(f"  {m:<13} {stats[m]['avg_words']:>9.0f} {compression:>11.0f}% {stats[m]['avg_time']:>9.1f}s")

# Answer length comparison
print(f"\n{'Method':<15} {'Avg Answer Len':>15}")
print("-" * 35)
for m in methods:
    answer_lens = [word_count(row["results"][m]["answer"]) for row in eval_results]
    print(f"  {m:<13} {sum(answer_lens)/len(answer_lens):>14.0f}")

## 22. Quality Comparison — LLM Judge Across All Questions

In [ ]:
print("QUALITY COMPARISON: Naive vs Abstractive Compressed")
print("=" * 80)

wins = {"naive": 0, "abstractive": 0, "tie": 0}
quality_scores = {"naive": [], "abstractive": []}

for row in eval_results:
    j = judge_answers(
        row["question"],
        row["results"]["naive"],
        row["results"]["abstractive"],
    )

    winner = "naive" if j.get("winner") == "A" else "abstractive" if j.get("winner") == "B" else "tie"
    wins[winner] += 1

    qa = j.get("quality_a", 3)
    qb = j.get("quality_b", 3)
    if isinstance(qa, (int, float)):
        quality_scores["naive"].append(qa)
    if isinstance(qb, (int, float)):
        quality_scores["abstractive"].append(qb)

    q_short = textwrap.shorten(row["question"], 55, placeholder="...")
    print(f"  {q_short:<55} → {winner}")

print(f"\n  WINS: naive={wins['naive']}, abstractive={wins['abstractive']}, tie={wins['tie']}")

if quality_scores["naive"]:
    avg_n = sum(quality_scores["naive"]) / len(quality_scores["naive"])
    avg_a = sum(quality_scores["abstractive"]) / len(quality_scores["abstractive"])
    print(f"  AVG QUALITY: naive={avg_n:.1f}/5, abstractive={avg_a:.1f}/5")

---

# Part F — Analysis & Learning Notes

## 23. When to Use Each Strategy

| Situation | Best Strategy | Why |
|-----------|--------------|-----|
| Low latency required, low risk | **Rerank only** | Fast, no LLM overhead for compression |
| Long passages, query is specific | **Extractive** | Keeps only relevant sentences, no LLM cost |
| Complex question, diverse passages | **Abstractive** | LLM distills key info query-relevantly |
| High-stakes (medical, legal) | **Abstractive + verify** | Maximum quality, hallucination check on compressed text |
| Prototype / MVP | **Naive** | Simplest, baseline to beat |

### Trade-offs

```
                  Speed          Quality        Token Savings
  Naive:          ██████████     ██████         (none)
  Reranked:       ████████       ████████       ██
  Extractive:     ███████        ████████       ██████
  Abstractive:    ███            █████████      ████████
```

### The Compression-Accuracy Trade-off

Too much compression loses information. Too little wastes tokens. The sweet spot depends on:
- **Context window size** — small windows need aggressive compression
- **Query complexity** — specific queries allow more compression; broad queries need more context
- **Passage redundancy** — if passages repeat information, compression removes duplication

## 24. Common Mistakes

| Mistake | Why It's Wrong | Better Approach |
|---------|---------------|----------------|
| Using only embedding similarity for ranking | Bi-encoders miss word-level interactions | Add cross-encoder reranking |
| Reranking all passages in the database | Cross-encoders are O(n) per query | Retrieve top-100 first, rerank that set |
| Compressing already-short chunks | Overhead exceeds savings on small texts | Only compress chunks above a word threshold |
| Same embedding model for retrieval and reranking | These are fundamentally different tasks | Use bi-encoder for retrieval, cross-encoder for reranking |
| Ignoring compression artifacts | LLM compression can itself introduce hallucinations | Verify compressed text against original |
| Hard-coding similarity threshold | Optimal threshold varies by domain and model | Tune threshold on validation queries |

## 25. Production Improvement Ideas

1. **Adaptive compression depth** — compress more aggressively for simple queries, less for complex ones
2. **Hybrid compression** — extractive first (fast), then abstractive on extracted sentences only
3. **Cross-encoder fine-tuning** — train the reranker on your domain Q&A pairs
4. **ColBERT-style late interaction** — faster than full cross-encoder while more accurate than bi-encoder
5. **Caching compressed results** — same query+passage pair always compresses the same; cache it
6. **Compression ratio monitoring** — track how much context you save per query in production
7. **Passage deduplication** — near-duplicate passages waste compression effort; deduplicate first

## 26. Exercises

### Exercise 1
Implement **hybrid compression**: first run extractive compression (fast, no LLM) to reduce each passage to ~50% of words, then run abstractive compression on the extracted sentences only. Compare the quality and speed to pure abstractive compression.

### Exercise 2
Build a **compression quality checker**: for each compressed passage, verify that all facts in the compressed version appear in the original. Flag any "compression hallucinations" where the LLM adds information during abstractive compression.

### Exercise 3
Experiment with **different reranking models**: try `cross-encoder/ms-marco-TinyBERT-L-2-v2` (faster, less accurate) and `cross-encoder/ms-marco-MiniLM-L-12-v2` (bigger, more accurate). Compare their impact on answer quality and latency.

### Mini Challenge
Build an **adaptive pipeline** that dynamically chooses the compression strategy: if the reranker's top-5 scores are all high (>5.0), skip compression (all passages are relevant). If scores vary widely, use extractive compression. If the query is complex, use abstractive. Measure improvement over a fixed strategy.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Knowledge base: {len(PASSAGES)} passages, {len(set(p['topic'] for p in PASSAGES))} topics")
print(f"Evaluation:     {len(EVAL_QUESTIONS)} questions, 4 pipeline variants")
print()
print(f"Pipeline variants tested:")
print(f"  1. Naive       — top-{5} by embedding only")
print(f"  2. Reranked    — top-{RETRIEVE_K} → cross-encoder rerank → top-{RERANK_TOP}")
print(f"  3. Extractive  — reranked + keep relevant sentences by similarity")
print(f"  4. Abstractive — reranked + LLM-compress each passage")
print()
print(f"Context efficiency:")
for m in methods:
    compression = stats[m]["avg_words"] / naive_avg * 100
    print(f"  {m:<13} avg {stats[m]['avg_words']:.0f} words ({compression:.0f}% of naive)")
print()
print(f"Components built:")
print(f"  1. Bi-encoder retriever     (MiniLM, fast, wide net)")
print(f"  2. Cross-encoder reranker   (ms-marco, accurate reordering)")
print(f"  3. Extractive compressor    (sentence-level similarity filter)")
print(f"  4. Abstractive compressor   (LLM query-focused summarization)")
print(f"  5. Score-based filter       (threshold on rerank scores)")
print(f"  6. LLM-as-judge evaluator   (head-to-head quality comparison)")

## 27. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Retrieval rank ≠ true relevance** — embedding similarity is approximate; cross-encoder reranking fixes ordering errors |
| 2 | **More chunks ≠ better answers** — irrelevant context dilutes attention and wastes tokens |
| 3 | **Retrieve wide, rerank narrow** — cast a wide net (top-15/20), then prune aggressively with a cross-encoder |
| 4 | **Extractive compression is free** — sentence-level filtering needs only embeddings, no LLM calls |
| 5 | **Abstractive compression gives the best quality** — but adds latency and can introduce compression artifacts |
| 6 | **Cross-encoders process (query, doc) jointly** — this is fundamentally more accurate than bi-encoder dot products |
| 7 | **Context compression can cut tokens by 40-70%** while maintaining or improving answer quality |
| 8 | **Compression is especially valuable for large context windows** — just because you can fit 100k tokens doesn't mean you should |
| 9 | **"Lost in the middle" is real** — LLMs underweight information in the middle of long contexts; compression concentrates the signal |
| 10 | **The right strategy depends on your constraints** — latency-sensitive: extractive; quality-sensitive: abstractive; budget-sensitive: rerank only |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*